In [1]:
# stress test

In [ ]:
import pandas as pd
import numpy as np

In [ ]:
n = 100

df = pd.DataFrame({

    "account_id": range(n),

    "symbol": np.random.choice(
        ["BTC", "ETH", "SOL"],
        n
    ),

    "price": np.random.randint(
        100,
        50000,
        n
    ),

    "position": np.random.randint(
        1,
        50,
        n
    ),

    "balance": np.random.randint(
        1000,
        20000,
        n
    ),

    "leverage": np.random.choice(
        [5, 10, 20, 50],
        n
    )

})

df.head()

,account_id,symbol,price,position,balance,leverage
0,0,SOL,22654,24,12736,50
1,1,BTC,45092,2,7340,50
2,2,SOL,4768,29,1494,5
3,3,ETH,13159,46,13497,20
4,4,BTC,24579,14,19054,50


In [ ]:
df["notional"] = df["price"] * df["position"]

In [ ]:
df["initial_margin"] = df["notional"] / df["leverage"]    #initial_margin = notional / leverage

In [ ]:
df["mm_rate"] = 0.05

df["maintenance_margin"] = (
    df["notional"] * df["mm_rate"]
)  #maintenance margin = notional * rate

In [ ]:
df["margin_ratio"] = (
    df["balance"] / df["notional"]
)

df.head()

,account_id,symbol,price,position,balance,leverage,notional,initial_margin,mm_rate,maintenance_margin,margin_ratio
0,0,SOL,22654,24,12736,50,543696,10873.92,0.05,27184.8,0.023425
1,1,BTC,45092,2,7340,50,90184,1803.68,0.05,4509.2,0.081389
2,2,SOL,4768,29,1494,5,138272,27654.40,0.05,6913.6,0.010805
3,3,ETH,13159,46,13497,20,605314,30265.70,0.05,30265.7,0.022298
4,4,BTC,24579,14,19054,50,344106,6882.12,0.05,17205.3,0.055372


In [ ]:
df["liquidation"] = (
    df["balance"] < df["maintenance_margin"]
).astype(int)

df["liquidation"].value_counts()   #balance < maintenance_margin

,count
liquidation,
1,69
0,31


In [ ]:
df["liq_price"] = (
    df["price"]
    * (1 - df["balance"] / df["notional"])
)    #liq price ≈ price * (1 - balance/notional)

In [ ]:
liq_accounts = df[
    df["liquidation"] == 1
]

liq_accounts

,account_id,symbol,price,position,balance,leverage,notional,initial_margin,mm_rate,maintenance_margin,margin_ratio,liquidation,liq_price
0,0,SOL,22654,24,12736,50,543696,10873.92,0.05,27184.8,0.023425,1,22123.333333
2,2,SOL,4768,29,1494,5,138272,27654.40,0.05,6913.6,0.010805,1,4716.482759
3,3,ETH,13159,46,13497,20,605314,30265.70,0.05,30265.7,0.022298,1,12865.586957
5,5,ETH,24756,10,2397,10,247560,24756.00,0.05,12378.0,0.009683,1,24516.300000
7,7,SOL,43775,40,15617,50,1751000,35020.00,0.05,87550.0,0.008919,1,43384.575000
...,...,...,...,...,...,...,...,...,...,...,...,...,...
93,93,SOL,24820,49,18880,10,1216180,121618.00,0.05,60809.0,0.015524,1,24434.693878
94,94,ETH,22012,22,7980,5,484264,96852.80,0.05,24213.2,0.016479,1,21649.272727
95,95,SOL,31559,46,6050,20,1451714,72585.70,0.05,72585.7,0.004167,1,31427.478261
96,96,BTC,35759,26,6699,5,929734,185946.80,0.05,46486.7,0.007205,1,35501.346154


In [ ]:
shock = 0.8

df["shock_price"] = df["price"] * shock

df["shock_notional"] = (
    df["shock_price"] * df["position"]
)   #-20%shock

In [ ]:
df["shock_liq"] = (
    df["balance"]
    < df["shock_notional"] * df["mm_rate"]
).astype(int)

df["shock_liq"].value_counts()     #stress test

,count
shock_liq,
1,64
0,36


In [ ]:
insurance_loss = df[
    df["shock_liq"] == 1
]["shock_notional"].sum()

insurance_loss #insurance fund

np.float64(43270590.400000006)

In [ ]:
print("Total accounts:", len(df))

print("Liquidations:", df["liquidation"].sum())

print("Shock liquidations:", df["shock_liq"].sum())

print("Insurance loss:", insurance_loss)

Total accounts: 100
Liquidations: 69
Shock liquidations: 64
Insurance loss: 43270590.400000006
